# 01 — EDA: Biohub Cell Tracking During Development

Zebra balığı embriyosunda 3B + zaman mikroskop verisinin keşfi.

**Hedefler (Hafta 1):**
- Gerçek `/kaggle/input` veri yolunu **otomatik bul** (yol haritasındaki yol tahminî).
- Bir train örneğini yükle: `.zarr` görüntü + `.geff` ground-truth graf.
- `(T, Z, Y, X)` boyutlarını, dtype'ı, **anizotropik ölçeği** (Z=1.625, Y=X=0.40625 µm) doğrula.
- **Seyrek etiket** yoğunluğunu ölç (kare başına node sayısı).
- Node/edge/**bölünme** istatistikleri, track uzunlukları.
- 2B projeksiyon görselleri (Kaggle'da napari başsız) + GT node overlay.

> Metrik hatırlatma: eşleştirme toleransı **7 µm**, mesafeler µm cinsinden. Skor = %90 edge + %10 division.

## 0 — Veri yolunu keşfet (ÖNCE BUNU ÇALIŞTIR)

Yol haritası `/kaggle/input/competitions/...` diyor ama Kaggle yarışma verisi genelde
`/kaggle/input/<competition-slug>/` altına bağlanır. Bu hücre varsayım yapmadan `/kaggle/input`
ağacını tarar, `.zarr` ve `.geff` dosyalarını bulur ve `DATA_ROOT` / `TRAIN_DIR` / `TEST_DIR`
değişkenlerini otomatik ayarlar.

In [ ]:
import os
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

# 1) /kaggle/input altindaki tum bagli dataset'leri goster
print('=== /kaggle/input icerigi ===')
if INPUT_ROOT.exists():
    for p in sorted(INPUT_ROOT.iterdir()):
        print(' ', p.name + ('/' if p.is_dir() else ''))
else:
    print('  (yok — Kaggle disinda calisiyor olabilirsin)')

# 2) .zarr ve .geff dizinlerini/dosyalarini bul (zarr aslinda bir dizindir)
def find_paths(root, suffix, limit=8):
    hits = []
    if not root.exists():
        return hits
    for dirpath, dirnames, filenames in os.walk(root):
        # .zarr / .geff dizin olarak gelir -> dirnames'te ara
        for d in list(dirnames):
            if d.endswith(suffix):
                hits.append(Path(dirpath) / d)
                dirnames.remove(d)  # icine inme, cok yavaslar
        for f in filenames:
            if f.endswith(suffix):
                hits.append(Path(dirpath) / f)
        if len(hits) >= limit:
            break
    return hits

zarrs = find_paths(INPUT_ROOT, '.zarr')
geffs = find_paths(INPUT_ROOT, '.geff')
print('\n=== bulunan .zarr (ilk 8) ===')
for p in zarrs: print(' ', p)
print('\n=== bulunan .geff (ilk 8) ===')
for p in geffs: print(' ', p)

In [ ]:
# DATA_ROOT'u otomatik cikar: bir .zarr'in ust dizini genelde train/ veya test/
# Yapi: <DATA_ROOT>/train/<sample>.zarr + <sample>.geff  ,  <DATA_ROOT>/test/<sample>.zarr
# DOGRULANDI (2026-07): gercek yol asagidaki gibi (yol haritasi dogruymus).
COMP_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')

def guess_split_dir(paths, name):
    for p in paths:
        if f'/{name}/' in str(p) or p.parent.name == name:
            return p.parent
    return None

TRAIN_DIR = guess_split_dir(zarrs, 'train') or guess_split_dir(geffs, 'train')
TEST_DIR  = guess_split_dir(zarrs, 'test')
# Fallback: dogrulanmis sabit yol (auto-detect bos donerse)
if TRAIN_DIR is None and (COMP_DIR / 'train').exists(): TRAIN_DIR = COMP_DIR / 'train'
if TEST_DIR is None and (COMP_DIR / 'test').exists():  TEST_DIR = COMP_DIR / 'test'

DATA_ROOT = TRAIN_DIR.parent if TRAIN_DIR is not None else (zarrs[0].parent if zarrs else COMP_DIR)

print('DATA_ROOT :', DATA_ROOT)
print('TRAIN_DIR :', TRAIN_DIR)
print('TEST_DIR  :', TEST_DIR)

# Train ornek ciftlerini listele (.zarr <-> .geff eslesmesi)
def list_samples(d, require_tracks=True):
    if d is None or not d.exists():
        return []
    samples = []
    for z in sorted(d.glob('*.zarr')):
        g = z.with_suffix('.geff')
        samples.append({'name': z.stem, 'image': z, 'tracks': g if g.exists() else None})
    return samples

train_samples = list_samples(TRAIN_DIR)
test_samples  = list_samples(TEST_DIR)
print(f'\n{len(train_samples)} train ornegi, {len(test_samples)} test ornegi.')

# Embriyo/film onekleri (ornek adi: <prefix>_<hash>) — CV'yi buna gore boldur (leakage!)
from collections import Counter
prefixes = Counter(s['name'].split('_')[0] for s in train_samples)
print('train film onekleri:', dict(prefixes))

# DIKKAT: test klipleri ayni isimle train'de de olabilir (ayni filmin farkli penceresi).
train_names = {s['name'] for s in train_samples}
overlap = sorted(s['name'] for s in test_samples if s['name'] in train_names)
print(f'train ile ayni isimli test ornegi: {len(overlap)} ->', overlap[:8])
for s in train_samples[:10]:
    print('  ', s['name'], '| tracks:', 'var' if s['tracks'] else 'YOK')

## 1 — Ortam & kütüphaneler

Kaggle'da çoğu paket kurulu değil. `zarr` genelde var; `geff`/`tracksdata` gerekebilir.
Aşağıdaki hücre eksikse kurar (internet açık bir notebook gerekir).

In [ ]:
# ORTAM NOTU: Bu bir CODE COMPETITION — submit icin Internet KAPALI olmali.
# Yarisma imajinda zarr 3.2.1, numcodecs, skimage, scipy, networkx, polars hazir geliyor.
# pip install YAPMA: internet gerektirir ve submit'i bloklar.
import numpy as np
import matplotlib.pyplot as plt
import zarr
print('zarr', zarr.__version__)

# ultrack / tracksdata / geff / monai bu ortamda YOK ve kurulamaz.
# GEFF'i duz zarr ile okuyoruz (bkz. 4. bolum) — ek paket gerekmiyor.

## 2 — Görüntüyü yükle (OME-Zarr)

Görüntü, `.zarr` dizini içinde `0/` yolunda tek bir dizi (array). Beklenen boyut `(T, Z, Y, X)`,
dtype `uint16`. Zarr sayesinde **tümünü RAM'e almadan** dilim dilim okuyabiliriz.

In [ ]:
# Anizotropik voxel olcegi (yaris sabiti) — mesafeler bununla µm'ye cevrilir
SCALE_ZYX = np.array([1.625, 0.40625, 0.40625])  # (Z, Y, X) µm/piksel
MATCH_TOL_UM = 7.0

assert train_samples, 'Once yol kesif hucrelerini calistir (DATA_ROOT bulunamadi).'
sample = train_samples[0]
print('Ornek:', sample['name'])

def open_image(zarr_path):
    """OME-Zarr goruntuyu ac. Once '0' alt-dizisini, sonra kok diziyi dener."""
    grp = zarr.open(str(zarr_path), mode='r')
    if hasattr(grp, 'shape'):
        return grp  # dogrudan array
    for key in ['0', 's0', 'image']:
        if key in grp:
            return grp[key]
    # ilk array'i bul
    for k in grp.array_keys():
        return grp[k]
    raise ValueError(f'{zarr_path} icinde array bulunamadi. Anahtarlar: {list(grp.keys())}')

img = open_image(sample['image'])
print('shape (T,Z,Y,X):', img.shape)
print('dtype           :', img.dtype)
print('chunks          :', getattr(img, 'chunks', None))
T, Z, Y, X = img.shape
print(f'\nFiziksel hacim ~ Z:{Z*SCALE_ZYX[0]:.0f}µm  Y:{Y*SCALE_ZYX[1]:.0f}µm  X:{X*SCALE_ZYX[2]:.0f}µm, {T} kare')

In [ ]:
# Tek bir zaman karesinin yogunluk istatistikleri (tam hacmi degil, tek kareyi oku)
t0 = T // 2
vol = np.asarray(img[t0])  # (Z, Y, X)
print(f'Kare t={t0}: shape {vol.shape}, dtype {vol.dtype}')
print(f'min {vol.min()}  max {vol.max()}  mean {vol.mean():.1f}  median {np.median(vol):.1f}')
for q in [50, 90, 99, 99.9]:
    print(f'  p{q}: {np.percentile(vol, q):.1f}')

plt.figure(figsize=(6,3))
plt.hist(vol[vol > 0].ravel(), bins=100, log=True)
plt.title(f'{sample["name"]}  t={t0}  yogunluk (log)')
plt.xlabel('deger'); plt.ylabel('adet (log)')
plt.tight_layout(); plt.show()

## 3 — 2B projeksiyonlar (napari yerine)

Kaggle'da napari başsız. 3B hacmi anlamak için **maksimum yoğunluk projeksiyonu (MIP)**
kullanıyoruz: Z ekseni boyunca max → XY görüntüsü. Görselleri `outputs/figures`'a kaydet.

In [ ]:
FIG_DIR = Path('/kaggle/working')  # Kaggle'da indirilebilir cikti klasoru

def mip_panels(vol, name, t):
    """XY, XZ, YZ maksimum-yogunluk projeksiyonlari."""
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    ax[0].imshow(vol.max(0), cmap='gray'); ax[0].set_title('XY (Z-MIP)')
    ax[1].imshow(vol.max(1), cmap='gray', aspect=SCALE_ZYX[0]/SCALE_ZYX[2]); ax[1].set_title('XZ (Y-MIP)')
    ax[2].imshow(vol.max(2), cmap='gray', aspect=SCALE_ZYX[0]/SCALE_ZYX[1]); ax[2].set_title('YZ (X-MIP)')
    for a in ax: a.axis('off')
    fig.suptitle(f'{name}  t={t}  MIP')
    fig.tight_layout()
    out = FIG_DIR / f'mip_{name}_t{t}.png'
    fig.savefig(out, dpi=110, bbox_inches='tight')
    print('kaydedildi:', out)
    plt.show()

mip_panels(vol, sample['name'], t0)

In [ ]:
# Zamanla degisim: birkac kareden XY-MIP montaji (hareketi gozle gor)
ts = np.linspace(0, T - 1, min(T, 6), dtype=int)
fig, axes = plt.subplots(1, len(ts), figsize=(3*len(ts), 3))
for a, t in zip(np.atleast_1d(axes), ts):
    a.imshow(np.asarray(img[t]).max(0), cmap='gray'); a.set_title(f't={t}'); a.axis('off')
fig.suptitle(f'{sample["name"]} — zaman boyunca XY-MIP')
fig.tight_layout()
fig.savefig(FIG_DIR / f'time_montage_{sample["name"]}.png', dpi=110, bbox_inches='tight')
plt.show()

## 4 — Ground-truth track grafını yükle (GEFF)

GEFF = zarr tabanlı seyrek graf. **Node** = `(t, z, y, x)` hücre merkezi, **edge** = `t → t+1`
bağlantısı, **bölünme** = bir ebeveyn → iki çocuk. Yükleme için birkaç yol deniyoruz
(`geff` → `tracksdata` → ham zarr). Amaç: node/edge koordinatlarını bir tabloya çıkarmak.

In [ ]:
import networkx as nx

def load_geff_graph(geff_path):
    """GEFF'i networkx grafina cevir. Node attr: t,z,y,x. Coklu backend dener."""
    geff_path = str(geff_path)
    # 1) geff paketi
    try:
        import geff
        for fn in ['read_nx', 'read_networkx', 'read']:
            if hasattr(geff, fn):
                g = getattr(geff, fn)(geff_path)
                g = g[0] if isinstance(g, tuple) else g
                if isinstance(g, nx.Graph):
                    return g, 'geff.' + fn
    except Exception as e:
        print('geff denendi, olmadi:', e)
    # 2) tracksdata
    try:
        from tracksdata.graph import RustWorkXGraph
        g = RustWorkXGraph.from_geff(geff_path)
        return g, 'tracksdata.RustWorkXGraph'
    except Exception as e:
        print('tracksdata denendi, olmadi:', e)
    # 3) ham zarr — yapiyi incele
    grp = zarr.open(geff_path, mode='r')
    print('GEFF ham zarr yapisi (backend yok, elle parse):')
    print(grp.tree() if hasattr(grp, 'tree') else list(grp.keys()))
    return grp, 'raw-zarr'

assert sample['tracks'] is not None, 'Bu ornekte .geff yok'
graph, backend = load_geff_graph(sample['tracks'])
print('\nyuklendi. backend =', backend, '| tip =', type(graph))

In [ ]:
# GEFF ham zarr ise node/edge dizilerini elle cikar (en tasinabilir yol).
# GEFF speci: <geff>/nodes/{ids, props/{t,z,y,x,...}} ve <geff>/edges/{ids, props/...}
import polars as pl

def geff_to_tables(geff_path):
    g = zarr.open(str(geff_path), mode='r')
    def read_props(group):
        node_props = {}
        props = group['props'] if 'props' in group else group
        for k in props.keys():
            arr = props[k]
            # her prop 'values' alt dizisi tasiyabilir
            if hasattr(arr, 'keys') and 'values' in arr:
                node_props[k] = np.asarray(arr['values'])
            elif hasattr(arr, 'shape'):
                node_props[k] = np.asarray(arr)
        return node_props
    nodes = read_props(g['nodes'])
    node_ids = np.asarray(g['nodes']['ids']) if 'ids' in g['nodes'] else np.arange(len(next(iter(nodes.values()))))
    nodes_df = pl.DataFrame({'node_id': node_ids, **{k: v for k, v in nodes.items()}})
    edges_arr = np.asarray(g['edges']['ids'])  # (E, 2) genelde
    edges_df = pl.DataFrame({'src': edges_arr[:, 0], 'dst': edges_arr[:, 1]})
    return nodes_df, edges_df

if backend == 'raw-zarr':
    nodes_df, edges_df = geff_to_tables(sample['tracks'])
elif backend.startswith('geff'):
    # networkx -> tablo
    recs = [{'node_id': n, **d} for n, d in graph.nodes(data=True)]
    nodes_df = pl.DataFrame(recs)
    edges_df = pl.DataFrame({'src': [u for u, v in graph.edges()], 'dst': [v for u, v in graph.edges()]})
else:  # tracksdata
    nodes_df = graph.node_attrs().to_polars() if hasattr(graph, 'node_attrs') else None
    edges_df = graph.edge_attrs().to_polars() if hasattr(graph, 'edge_attrs') else None

print('NODES', nodes_df.shape)
print(nodes_df.head())
print('\nEDGES', edges_df.shape)
print(edges_df.head())

## 5 — Seyrek etiket yoğunluğu (KRİTİK)

Ground-truth **seyrek**: her karede hücrelerin sadece bir alt kümesi etiketli. Kare başına
node sayısını çıkarıp bunu görüntüdeki tahmini hücre sayısıyla kıyaslamak eğitim/eval
stratejimizi belirler.

In [ ]:
# 't' kolonunun adini bul (t / time / frame gibi)
tcol = next((c for c in nodes_df.columns if c.lower() in ('t', 'time', 'frame')), None)
print('zaman kolonu:', tcol)
per_t = nodes_df.group_by(tcol).len().sort(tcol)
print(per_t)

counts = per_t['len'].to_numpy()
print(f'\nKare basina node: min {counts.min()}  max {counts.max()}  ort {counts.mean():.1f}')
print(f'Toplam node: {nodes_df.height}  |  toplam edge: {edges_df.height}')

plt.figure(figsize=(8,3))
plt.bar(per_t[tcol].to_numpy(), counts)
plt.xlabel('t (kare)'); plt.ylabel('etiketli node')
plt.title(f'{sample["name"]} — kare basina GT node (seyreklik)')
plt.tight_layout(); plt.savefig(FIG_DIR / f'gt_density_{sample["name"]}.png', dpi=110); plt.show()

## 6 — Bölünmeler ve track uzunlukları

Bölünme = out-degree 2 olan node (bir ebeveyn → iki çocuk). Track = bir hücrenin zaman
boyunca zinciri. Skorun %10'u bölünmelerden gelir.

In [ ]:
# Yonlu graf kur (src -> dst = t -> t+1)
G = nx.DiGraph()
G.add_nodes_from(nodes_df['node_id'].to_list())
G.add_edges_from(zip(edges_df['src'].to_list(), edges_df['dst'].to_list()))

out_deg = np.array([d for _, d in G.out_degree()])
in_deg = np.array([d for _, d in G.in_degree()])
n_div = int((out_deg == 2).sum())
n_start = int((in_deg == 0).sum())   # track baslangici (ilk kare veya yeni giren)
n_end = int((out_deg == 0).sum())    # track sonu (son kare veya kaybolan)
print(f'Bolunme (out-deg==2): {n_div}')
print(f'Track baslangici (in-deg==0): {n_start}')
print(f'Track sonu (out-deg==0): {n_end}')
print(f'Anomali out-deg>2: {(out_deg > 2).sum()} | in-deg>1: {(in_deg > 1).sum()}')

# Zayif bagli bilesenler ~ soy agaci sayisi; her bilesenin node sayisi ~ track uzunlugu
comp_sizes = [len(c) for c in nx.weakly_connected_components(G)]
print(f'\nSoy agaci (bilesen) sayisi: {len(comp_sizes)}')
print(f'Bilesen boyu: min {min(comp_sizes)} max {max(comp_sizes)} ort {np.mean(comp_sizes):.1f}')
plt.figure(figsize=(7,3))
plt.hist(comp_sizes, bins=40); plt.xlabel('bilesendeki node sayisi'); plt.ylabel('adet')
plt.title('Track / soy agaci uzunluk dagilimi'); plt.tight_layout()
plt.savefig(FIG_DIR / f'track_len_{sample["name"]}.png', dpi=110); plt.show()

## 7 — Kare-arası hareket (linking için hayati)

Hücreler `t → t+1` arası ne kadar hareket ediyor (µm)? Bu, linking'te mesafe eşiğini ve
hareket modelini belirler. Anizotropi nedeniyle mesafeyi **µm** cinsinden hesaplıyoruz.

In [ ]:
# node_id -> (z,y,x) koordinat sozlugu (kolon adlarini esnek bul)
def col(df, *names):
    return next((c for c in df.columns if c.lower() in names), None)
zc, yc, xc = col(nodes_df,'z'), col(nodes_df,'y'), col(nodes_df,'x')
pos = {r['node_id']: np.array([r[zc], r[yc], r[xc]]) for r in nodes_df.iter_rows(named=True)}

disp_um = []
for u, v in G.edges():
    if u in pos and v in pos:
        disp_um.append(np.linalg.norm((pos[v] - pos[u]) * SCALE_ZYX))
disp_um = np.array(disp_um)
print(f'Kare-arasi yer degistirme (µm): ort {disp_um.mean():.2f}  medyan {np.median(disp_um):.2f}  p95 {np.percentile(disp_um,95):.2f}  max {disp_um.max():.2f}')
print(f'7 µm toleransini asan edge orani: {(disp_um > MATCH_TOL_UM).mean()*100:.1f}%')

plt.figure(figsize=(7,3))
plt.hist(disp_um, bins=60); plt.axvline(MATCH_TOL_UM, color='r', ls='--', label='7 µm tol')
plt.xlabel('yer degistirme (µm)'); plt.ylabel('edge'); plt.legend()
plt.title('Kare-arasi hareket'); plt.tight_layout()
plt.savefig(FIG_DIR / f'motion_{sample["name"]}.png', dpi=110); plt.show()

## 8 — GT node'ları görüntü üzerine bindir (sanity check)

Etiketlerin gerçekten hücre merkezlerine oturduğunu gözle doğrula: bir karenin XY-MIP'ine
o kareye ait node'ları (XY projeksiyonu) kırmızı ile bindir.

In [ ]:
t_show = int(per_t[tcol][per_t['len'].arg_max()])  # en cok etiketli kare
vol_t = np.asarray(img[t_show])
sub = nodes_df.filter(pl.col(tcol) == t_show)
ys = sub[yc].to_numpy(); xs = sub[xc].to_numpy()

plt.figure(figsize=(7,7))
plt.imshow(vol_t.max(0), cmap='gray')
plt.scatter(xs, ys, s=18, facecolors='none', edgecolors='r', linewidths=0.8)
plt.title(f'{sample["name"]}  t={t_show}  —  {len(xs)} GT node (kirmizi)')
plt.axis('off'); plt.tight_layout()
plt.savefig(FIG_DIR / f'overlay_{sample["name"]}_t{t_show}.png', dpi=130, bbox_inches='tight')
plt.show()

## 9 — Tüm datasetler için özet tablo

Her train örneği için hızlı bir özet: görüntü boyutu, node/edge, bölünme, kare başına
yoğunluk. Ağır I/O yapmadan (sadece metadata + graf) çıkarılır.

In [ ]:
rows = []
for s in train_samples:
    try:
        im = open_image(s['image'])
        rec = {'name': s['name'], 'shape': str(tuple(im.shape)), 'dtype': str(im.dtype)}
        if s['tracks'] is not None:
            ndf, edf = geff_to_tables(s['tracks'])
            tc = next((c for c in ndf.columns if c.lower() in ('t','time','frame')), None)
            Gs = nx.DiGraph(); Gs.add_edges_from(zip(edf['src'].to_list(), edf['dst'].to_list()))
            odg = np.array([d for _, d in Gs.out_degree()]) if Gs.number_of_nodes() else np.array([])
            rec.update(nodes=ndf.height, edges=edf.height,
                       divisions=int((odg == 2).sum()),
                       nodes_per_t=round(ndf.height / ndf[tc].n_unique(), 1) if tc else None)
        rows.append(rec)
    except Exception as e:
        rows.append({'name': s['name'], 'shape': f'HATA: {e}'})

summary = pl.DataFrame(rows)
print(summary)
summary.write_csv(FIG_DIR / 'eda_summary.csv')
print('\nkaydedildi: eda_summary.csv')

## 10 — Notlar & sonraki adım

Bu hücreyi çalıştırdıktan sonra **kendi gözlemlerini** buraya yaz:

- Gerçek `DATA_ROOT` = ?  (yol haritasındaki tahminle karşılaştır)
- `(T, Z, Y, X)` = ?  · dtype = ?  · dataset sayısı = ?
- Kare başına ortalama GT node (seyreklik) = ?
- Kare-arası hareket p95 (µm) — linking eşiği için = ?
- Bölünme sayısı / oranı = ?
- Anomaliler (out-deg>2, eksik .geff, boş kareler) = ?

➡️ Sonraki: `02_baseline_ultrack.ipynb` — YOL A ile ilk gönderim.  
➡️ Sanity check (Hafta 1): metrik kodunu GT→GT verip **1.0** aldığını doğrula (`05_evaluation.ipynb`).